# Claim storage and conflict checking

Extract claims, save them for review, manually approve selected records, and check a new document against approved claims.

Run `uv sync` and select the project's `.venv` kernel. Setup loads `.env` without overriding existing environment variables. Live extraction and checking use `OPENAI_API_KEY` and `TYPESAFE_API_KEY` and make paid requests.

The examples below are fictional. They use a separate `.data/claims-demo` store. **Verified means manually approved by you**, not independently fact-checked. Running all cells never approves anything unless you explicitly fill in `approved_ids`.

In [ ]:
import html
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display

from data_platform.digest.claimify import extract_claims, parse_document
from data_platform.digest.claim_store import ClaimStore

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/data_platform").is_dir()
)
load_dotenv(project_root / ".env", override=False)


def show_table(rows, columns):
    if not rows:
        print("No rows.")
        return
    header = "".join(f"<th>{html.escape(label)}</th>" for label, key in columns)
    body = "".join(
        "<tr>" + "".join(
            f"<td style='white-space:pre-wrap'>{html.escape(str(row.get(key, '')))}</td>"
            for label, key in columns
        ) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def read_input(path, fallback, fallback_source):
    if path is None:
        return fallback, fallback_source
    path = Path(path).expanduser()
    if not path.is_absolute():
        path = project_root / path
    return path.read_text(encoding="utf-8"), str(path)


## Configure the store and models

Keep the demo store for these fictional examples. For your own documents, set `store_path` to `.data/claims` and set the input paths below. Markdown requires one H1 title and H2 section subtitles. Extraction and storage remain separate operations.

In [ ]:
store_path = project_root / ".data/claims-demo"
embedding_model = "openai:text-embedding-3-small"
comparison_model = "typesafe:jev-latest"
generation_model = "openai:gpt-6-luna"
max_concurrency = 4
top_k = 20

store = ClaimStore(
    store_path,
    embedding_model=embedding_model,
    comparison_model=comparison_model,
    max_concurrency=max_concurrency,
)
print("Store:", store.root)
print("Unverified:", len(store.list_claims("unverified")))
print("Verified:", len(store.list_claims("verified")))

## Extract a reference document

Leave `reference_path = None` to use the fictional example, or set it to a segmented Markdown file such as `.data/segmented/article-en-01.md`.

In [ ]:
reference_path = None
reference_markdown, reference_source = read_input(
    reference_path,
    """# Acme annual report

## Financial results

Acme's total revenue for 2025 was USD 10 million.

## Factory plans

Acme planned to open a Hanoi factory in 2025.
""",
    "demo:acme-annual-report",
)
reference_document = parse_document(reference_markdown)
print(reference_source)
print(reference_markdown)

In [ ]:
reference_result = await extract_claims(
    reference_markdown,
    decision_model=comparison_model,
    generation_model=generation_model,
    max_concurrency=max_concurrency,
)
print("Extracted claims:", len(reference_result.claims))

In [ ]:
reference_records = store.store_unverified(reference_result, source=reference_source)
show_table(
    [record.model_dump() for record in reference_records],
    [("ID", "id"), ("Claim", "text"), ("Original sentence", "original_sentence"),
     ("Section", "section_title"), ("Source", "source"), ("Start", "start"), ("End", "end")],
)
print("Review JSON files in:", store.root / "unverified")

## Manually approve selected claims

Read each JSON file and its source. Copy only the IDs you approve into `approved_ids`, then run the next cell. The empty default does nothing. You can instead move files from `unverified/` to `verified/` in your file manager.

Approval is your decision. Neither extraction, similarity, nor a comparison verdict approves a claim. Repeated imports preserve existing records. To withdraw approval, move the file out of `verified/` and synchronize again.

In [ ]:
approved_ids = []  # Paste reviewed IDs here, e.g. ["<full claim ID>"]

# Validate the whole selection before moving files. Never overwrite verified JSON.
promotions = []
for claim_id in dict.fromkeys(approved_ids):
    record = store.get_claim(claim_id)
    if record is None:
        raise ValueError(f"Unknown claim ID: {claim_id}")
    source_path = store.root / "unverified" / f"{claim_id}.json"
    target_path = store.root / "verified" / f"{claim_id}.json"
    if target_path.exists():
        print("Already approved:", claim_id)
        continue
    if not source_path.is_file():
        raise FileNotFoundError(source_path)
    promotions.append((source_path, target_path))

for source_path, target_path in promotions:
    source_path.rename(target_path)
    print("Approved:", target_path.stem)

verified = store.list_claims("verified")
print("Verified claims:", len(verified))
if not verified:
    print("Approve reference claims and rerun from this cell to get retrieval and comparison results.")

## Synchronize and retrieve

Only verified files enter the SQLite cache. New or edited claim text needs embeddings; metadata-only edits reuse vectors. Model changes rebuild vectors. `retrieve()` and `check()` also synchronize automatically, so the explicit sync below is for inspecting cache changes.

In [ ]:
sync = await store.sync_index()
print(sync.model_dump())

query = "Acme's total revenue for 2025 was USD 12 million."
matches = await store.retrieve(query, top_k=top_k)
show_table(
    [{**match.claim.model_dump(), "similarity": round(match.similarity, 4)} for match in matches],
    [("Similarity", "similarity"), ("Verified claim", "text"), ("ID", "id"), ("Source", "source")],
)

## Extract and check an incoming document

Use `incoming_path` for your own segmented Markdown. In this fictional example, the revenue assertion conflicts with the reference, while a plan to open a factory can coexist with its not opening. Model output can differ; inspect the source context.

In [ ]:
incoming_path = None
incoming_markdown, incoming_source = read_input(
    incoming_path,
    """# Acme follow-up report

## Financial results

Acme's total revenue for 2025 was USD 12 million.

## Factory status

Acme did not open a Hanoi factory in 2025.
""",
    "demo:acme-follow-up-report",
)
parse_document(incoming_markdown)
print(incoming_markdown)

incoming_result = await extract_claims(
    incoming_markdown,
    decision_model=comparison_model,
    generation_model=generation_model,
    max_concurrency=max_concurrency,
)
incoming_records = store.store_unverified(incoming_result, source=incoming_source)
print("Incoming claims:", len(incoming_records))

In [ ]:
report = await store.check(incoming_records, top_k=top_k)
print("Verified candidates available:", report.verified_count)
print("Pairs compared:", len(report.comparisons))
print("Verdicts:", dict(Counter(pair.decision.verdict for pair in report.comparisons)))
print(report.scope)
if not report.comparisons:
    print("No pairs were compared. Check that extraction produced claims and you approved reference records.")


def pair_rows(pairs):
    return [
        {
            "verdict": pair.decision.verdict,
            "similarity": round(pair.similarity, 4),
            "claim": pair.claim.text,
            "claim_id": pair.claim.id,
            "claim_source": pair.claim.source,
            "verified_claim": pair.verified_claim.text,
            "verified_id": pair.verified_claim.id,
            "verified_source": pair.verified_claim.source,
        }
        for pair in pairs
    ]


pair_columns = [
    ("Verdict", "verdict"), ("Similarity", "similarity"),
    ("Incoming claim", "claim"), ("Incoming ID", "claim_id"), ("Source", "claim_source"),
    ("Verified claim", "verified_claim"), ("Verified ID", "verified_id"), ("Verified source", "verified_source"),
]
print("Review first: conflicts and uncertain pairs")
show_table(pair_rows(report.review_required), pair_columns)

## Inspect all comparisons and save the report

Verdicts are `conflict`, `equivalent`, `compatible`, `unrelated`, and `uncertain`. Similarity is a retrieval score, not confidence in a verdict. Top-20 retrieval is not an exhaustive consistency check, and no detected conflict does not establish truth.

The exported report includes full source records, offsets, IDs, and all decisions. This does not modify approval status.

In [ ]:
show_table(pair_rows(report.comparisons), pair_columns)
report_path = store.root / "reports" / "latest-check.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report.model_dump_json(indent=2) + "\n", encoding="utf-8")
print("Saved report:", report_path)

## Inspect a stored record and the semantic evaluation cases

`get_claim(id)` returns a record or `None`; folder location determines approval. The evaluation cases below have manually reviewed expected labels, not measured live-model results. They cover quantities, negation, paraphrases, dates, attribution, missing context, plans, and unrelated assertions.

For cache rebuilding, remove only `index.sqlite` and run `await store.sync_index()`. Keep the JSON files: they are the source of truth. Invalid files and model failures raise errors rather than becoming “no conflict.”

In [ ]:
if incoming_records:
    print(store.get_claim(incoming_records[0].id).model_dump_json(indent=2))

cases_path = project_root / "tests/fixtures/claim_store/comparisons.json"
cases = json.loads(cases_path.read_text(encoding="utf-8"))
show_table(cases, [("Case", "case"), ("Claim", "claim"), ("Reference", "verified_claim"), ("Expected", "expected")])